# vLLM probe — free T4

Answers four questions the Keel build cannot answer on its own, none of which
need a cluster and none of which cost anything:

1. **What are vLLM's real Prometheus metric names?** The KEDA trigger in
   `control/render/manifests.py:_scaled_object` queries one nobody has ever
   observed. If it is wrong, lane C comes up healthy and silently never scales.
2. **Do our catalog engine args work?** `max-model-len: 8192`,
   `gpu-memory-utilization: 0.60`.
3. **What does an OOM actually say?** `control/provisioner/status.py` claims to
   detect it and offers advice; the message should match reality.
4. **Does a chat template apply cleanly?** The provisioner's smoke test assumes
   a real completion comes back.

**Runtime → Change runtime type → T4 GPU** before running anything.

Copy the SUMMARY block at the end back into the repo.

---

### Why this installs into a venv

The first attempt did `pip install vllm` into Colab's own environment. That
upgraded torch, which then mismatched Colab's **preinstalled torchaudio** —
and since `transformers` imports torchaudio unconditionally, vLLM could not get
through its own imports:

```
RuntimeError: Detected that PyTorch and torchaudio were compiled with
different CUDA versions
```

That is a packaging conflict, not a vLLM or T4 problem. Installing into an
isolated environment sidesteps Colab's preinstalled stack entirely. The
notebook kernel then only ever talks to the server over HTTP, so it needs none
of vLLM's dependencies itself.


In [1]:
!nvidia-smi --query-gpu=name,memory.total,compute_cap --format=csv


name, memory.total [MiB], compute_cap
Tesla T4, 15360 MiB, 7.5


## 1 · Install into an isolated environment

Several minutes — vLLM and its CUDA wheels are large. `uv` makes it noticeably
faster than pip.


In [2]:
!pip install -q uv
!uv venv /content/venv --python 3.12
!uv pip install --python /content/venv/bin/python -q vllm

VLLM = "/content/venv/bin/vllm"
PY   = "/content/venv/bin/python"
!{PY} -c "import vllm, torch; print('vllm', vllm.__version__, '| torch', torch.__version__, '| cuda', torch.version.cuda)"


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 20.1/20.1 MB 37.7 MB/s eta 0:00:00
Using CPython 3.12.14
Creating virtual environment at: venv
Activate with: source venv/bin/activate
vllm 0.28.0 | torch 2.13.0+cu130 | cuda 13.0


## 2 · Start with our catalog args, unmodified

Exactly what `catalog/qwen2.5-0.5b-instruct.yaml` specifies, plus `--dtype half`
because a T4 is compute capability 7.5 and has no bfloat16 — while Qwen2.5
declares bf16 in its config.

Startup is not instant: vLLM downloads weights, profiles memory and (on V1)
compiles. The loop below prints progress so a slow start is distinguishable
from a hung one.


In [3]:
import subprocess, time, requests

MODEL = "Qwen/Qwen2.5-0.5B-Instruct"
ARGS = [
    "--model", MODEL,
    "--port", "8000",
    "--max-model-len", "8192",           # catalog engine_args
    "--gpu-memory-utilization", "0.60",  # catalog engine_args
    "--dtype", "half",                   # T4: no bfloat16
]

log = open("/content/vllm.log", "w")
proc = subprocess.Popen([VLLM, "serve", *ARGS], stdout=log, stderr=subprocess.STDOUT)

t0, ready = time.time(), False
while time.time() - t0 < 1200:
    try:
        if requests.get("http://localhost:8000/health", timeout=2).status_code == 200:
            ready = True
            break
    except Exception:
        pass
    if proc.poll() is not None:
        print("SERVER EXITED -- last 40 lines:")
        print("".join(open("/content/vllm.log").readlines()[-40:]))
        break
    el = int(time.time() - t0)
    if el % 30 < 6:
        print(f"  {el:>4}s  still starting ...", flush=True)
    time.sleep(5)

LOAD_SECONDS = round(time.time() - t0, 1)
print(f"\nready={ready}  load_seconds={LOAD_SECONDS}")
if not ready:
    print("If it timed out rather than exited, re-run this cell with "
          "--enforce-eager appended to ARGS: it skips CUDA-graph capture, "
          "which is the slow part of startup on a T4.")


     0s  still starting ...
     5s  still starting ...
    30s  still starting ...
    35s  still starting ...
    60s  still starting ...
    65s  still starting ...
    90s  still starting ...
    95s  still starting ...
   120s  still starting ...
   125s  still starting ...

ready=True  load_seconds=130.1


In [4]:
# What vLLM said about dtype, memory and the KV cache while starting.
import re
for line in open("/content/vllm.log").read().splitlines():
    if re.search(r"dtype|bfloat16|float16|KV cache|GPU blocks|memory|Capturing", line, re.I):
        print(line[:200])


(APIServer pid=1567) INFO 09-01 20:24:46 [api_utils.py:272] non-default args: {'model_tag': 'Qwen/Qwen2.5-0.5B-Instruct', 'model': 'Qwen/Qwen2.5-0.5B-Instruct', 'dtype': 'half', 'max_model_len': 8192,
(APIServer pid=1567) WARNING 09-01 20:24:59 [model.py:2299] Casting torch.bfloat16 to torch.float16.
(EngineCore pid=1771) INFO 09-01 20:25:14 [core.py:122] Initializing a V1 LLM engine (v0.28.0) with config: model='Qwen/Qwen2.5-0.5B-Instruct', speculative_config=None, tokenizer='Qwen/Qwen2.5-0.5B-In
(EngineCore pid=1771) INFO 09-01 20:25:43 [model_runner.py:380] Model loading took 0.93 GiB memory and 24.724012 seconds
(EngineCore pid=1771) INFO 09-01 20:26:08 [gpu_worker.py:578] Available KV cache memory: 7.04 GiB
(EngineCore pid=1771) INFO 09-01 20:26:08 [kv_cache_utils.py:1869] GPU KV cache size: 614,960 tokens, Maximum concurrency for 8,192 tokens per request: 75.07x
Capturing CUDA graphs (FULL): 100%|██████████| 35/35 [00:01<00:00, 18.00it/s]
(EngineCore pid=1771) INFO 09-01 20:26:17

## 3 · The metric names — the most valuable output

Everything with a `vllm:` prefix. Compare against what `_scaled_object` queries
(`vllm:request_success_total`) and what `deployment_metrics` expects to feed on
(TTFT, TPOT, queue depth).


In [5]:
import requests, re
metrics = requests.get("http://localhost:8000/metrics", timeout=10).text
names = sorted({m.group(1) for m in re.finditer(r"^(vllm:[a-zA-Z0-9_]+)", metrics, re.M)})
print(f"{len(names)} vllm: metrics\n")
for n in names:
    print(" ", n)

print("\n--- the ones Keel depends on ---")
for want in ["vllm:request_success_total", "vllm:num_requests_waiting",
             "vllm:num_requests_running", "vllm:time_to_first_token_seconds",
             "vllm:time_per_output_token_seconds", "vllm:gpu_cache_usage_perc"]:
    print(f"  {'FOUND  ' if want in names else 'MISSING'} {want}")


96 vllm: metrics

  vllm:cache_config_info
  vllm:e2e_request_latency_seconds_bucket
  vllm:e2e_request_latency_seconds_count
  vllm:e2e_request_latency_seconds_created
  vllm:e2e_request_latency_seconds_sum
  vllm:engine_sleep_state
  vllm:estimated_flops_per_gpu_created
  vllm:estimated_flops_per_gpu_total
  vllm:estimated_read_bytes_per_gpu_created
  vllm:estimated_read_bytes_per_gpu_total
  vllm:estimated_write_bytes_per_gpu_created
  vllm:estimated_write_bytes_per_gpu_total
  vllm:external_prefix_cache_hits_created
  vllm:external_prefix_cache_hits_total
  vllm:external_prefix_cache_queries_created
  vllm:external_prefix_cache_queries_total
  vllm:generation_tokens_created
  vllm:generation_tokens_total
  vllm:inter_token_latency_seconds_bucket
  vllm:inter_token_latency_seconds_count
  vllm:inter_token_latency_seconds_created
  vllm:inter_token_latency_seconds_sum
  vllm:iteration_tokens_total_bucket
  vllm:iteration_tokens_total_count
  vllm:iteration_tokens_total_created
  vllm

## 4 · The smoke test, exactly as the provisioner runs it

`control/provisioner/main.py` sends this prompt and treats any reply as a pass.
Read the reply: a chat-template mismatch returns a perfectly valid 200 with
obviously wrong text, which is precisely what the smoke test is meant to catch
and could plausibly miss.


In [6]:
import requests, time
t0 = time.time()
r = requests.post("http://localhost:8000/v1/chat/completions", timeout=60, json={
    "model": MODEL,
    "messages": [{"role": "user", "content": "Reply with the single word: ok"}],
    "max_tokens": 16,
})
SMOKE = r.json()
print("status", r.status_code, f"in {time.time()-t0:.2f}s")
print("reply:", repr(SMOKE["choices"][0]["message"]["content"]))
print("usage:", SMOKE.get("usage"))


status 200 in 0.97s
reply: 'Ok'
usage: {'prompt_tokens': 36, 'total_tokens': 38, 'completion_tokens': 2, 'prompt_tokens_details': None, 'completion_tokens_details': None}


## 5 · Force an OOM and capture what it says

`control/provisioner/status.py` reports OOM with advice about accelerator size,
GPU count and `--max-model-len`. This checks the advice matches the real error.

Asks for a context far beyond what a 16GB T4 can hold a KV cache for.


In [7]:
import subprocess
proc.terminate(); proc.wait()

oom = subprocess.run(
    [VLLM, "serve", "--model", MODEL, "--port", "8001",
     "--max-model-len", "200000", "--gpu-memory-utilization", "0.95",
     "--dtype", "half"],
    capture_output=True, text=True, timeout=1200,
)
OOM_TEXT = oom.stdout + oom.stderr
print("exit", oom.returncode)
print("\n".join([l for l in OOM_TEXT.splitlines() if l.strip()][-25:]))


exit 1
(APIServer pid=2338)     async with build_async_engine_client(
(APIServer pid=2338)                ^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=2338)   File "/root/.local/share/uv/python/cpython-3.12.14-linux-x86_64-gnu/lib/python3.12/contextlib.py", line 210, in __aenter__
(APIServer pid=2338)     return await anext(self.gen)
(APIServer pid=2338)            ^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=2338)   File "/content/venv/lib/python3.12/site-packages/vllm/entrypoints/openai/api_server.py", line 127, in build_async_engine_client
(APIServer pid=2338)     async with build_async_engine_client_from_engine_args(
(APIServer pid=2338)                ^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=2338)   File "/root/.local/share/uv/python/cpython-3.12.14-linux-x86_64-gnu/lib/python3.12/contextlib.py", line 210, in __aenter__
(APIServer pid=2338)     return await anext(self.gen)
(APIServer pid=2338)            ^^^^^^^^^^^^^^^^^^^^^
(APIServer pid=2338)   File "/content/venv/lib/p

## 6 · Summary — paste this back


In [8]:
import json, subprocess
ver = subprocess.run([PY, "-c", "import vllm;print(vllm.__version__)"],
                     capture_output=True, text=True).stdout.strip()
print(json.dumps({
    "gpu": "T4 (compute 7.5, 16GB)",
    "vllm_version": ver,
    "model": MODEL,
    "load_seconds_cold": LOAD_SECONDS,
    "dtype_note": "bfloat16 unsupported on T4; --dtype half required",
    "metric_names": names,
    "keel_depends_on_present": {
        w: (w in names) for w in [
            "vllm:request_success_total", "vllm:num_requests_waiting",
            "vllm:time_to_first_token_seconds", "vllm:time_per_output_token_seconds",
        ]
    },
    "smoke_reply": SMOKE["choices"][0]["message"]["content"],
    "oom_signature": [l for l in OOM_TEXT.splitlines()
                      if any(k in l.lower() for k in
                             ["memory", "kv cache", "reduce", "oom", "valueerror"])][-6:],
}, indent=2))


{
  "gpu": "T4 (compute 7.5, 16GB)",
  "vllm_version": "0.28.0",
  "model": "Qwen/Qwen2.5-0.5B-Instruct",
  "load_seconds_cold": 130.1,
  "dtype_note": "bfloat16 unsupported on T4; --dtype half required",
  "metric_names": [
    "vllm:cache_config_info",
    "vllm:e2e_request_latency_seconds_bucket",
    "vllm:e2e_request_latency_seconds_count",
    "vllm:e2e_request_latency_seconds_created",
    "vllm:e2e_request_latency_seconds_sum",
    "vllm:engine_sleep_state",
    "vllm:estimated_flops_per_gpu_created",
    "vllm:estimated_flops_per_gpu_total",
    "vllm:estimated_read_bytes_per_gpu_created",
    "vllm:estimated_read_bytes_per_gpu_total",
    "vllm:estimated_write_bytes_per_gpu_created",
    "vllm:estimated_write_bytes_per_gpu_total",
    "vllm:external_prefix_cache_hits_created",
    "vllm:external_prefix_cache_hits_total",
    "vllm:external_prefix_cache_queries_created",
    "vllm:external_prefix_cache_queries_total",
    "vllm:generation_tokens_created",
    "vllm:generation_